In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("/content/train.csv")

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
df.shape

(891, 12)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [8]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [9]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df["PassengerId"].duplicated().sum()

np.int64(0)

In [13]:
print("Sex:",df["Sex"].unique())
print("Embarked:",df["Embarked"].unique())
print("Pclass:",df["Pclass"].unique())
print("Survived:",df["Survived"].unique())

Sex: ['male' 'female']
Embarked: ['S' 'C' 'Q' nan]
Pclass: [3 1 2]
Survived: [0 1]


In [10]:
df[["Age","Fare","SibSp","Parch"]].describe()

,Age,Fare,SibSp,Parch
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008,0.381594
std,14.526497,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,7.910400,0.000000,0.000000
50%,28.000000,14.454200,0.000000,0.000000
75%,38.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


In [11]:
df["Age"] = df["Age"].fillna(df["Age"].median())

In [12]:
df["Age"].isnull().sum()

np.int64(0)

In [13]:
df["Embarked"].value_counts()

,count
Embarked,
S,644
C,168
Q,77


In [14]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [15]:
df["Embarked"].isnull().sum()

np.int64(0)

In [16]:
df = df.drop(columns=["Cabin"])

In [17]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [18]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [19]:
print("Age below 0:", (df["Age"] < 0).sum())
print("Fare below 0:", (df["Fare"] < 0).sum())
print("Invalid Pclass:", (~df["Pclass"].isin([1, 2, 3])).sum())
print("Invalid Survived:", (~df["Survived"].isin([0, 1])).sum())

Age below 0: 0
Fare below 0: 0
Invalid Pclass: 0
Invalid Survived: 0


### Outlier Handling

Potential outliers in the Fare column were identified using the Interquartile Range (IQR) method. A total of 116 observations were flagged as potential outliers.

These observations were retained because unusually high fares may represent genuine ticket prices rather than data-entry errors. Removing them could result in loss of valid information.

In [20]:
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

outliers = df[(df["Fare"] < lower_bound) | (df["Fare"] > upper_bound)]

print("Number of Fare outliers:", len(outliers))

Lower bound: -26.724
Upper bound: 65.6344
Number of Fare outliers: 116


In [21]:
print("Rows:", df.shape[0])

Rows: 891


In [22]:
print("Columns:", df.shape[1])

Columns: 11


In [23]:
print("Missing values:", df.isnull().sum().sum())

Missing values: 0


In [24]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [27]:
df.to_csv("/content/cleaned_titanic.csv", index=False)

In [26]:
import os
print(os.path.exists("/content/cleaned_titanic.csv"))

True
